In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Windows 기본 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

#파일을 DataFrame으로 생성하고 객체 할당 
train_df=pd.read_csv(r'C:\Users\user\ESAA_YB\미니프로젝트1\movies_train.csv')
test_df=pd.read_csv(r"C:\Users\user\ESAA_YB\미니프로젝트1\movies_test.csv")

In [2]:
print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print('\n[test 컬럼 목록]')
print(test_df.columns.tolist())
print('\n[train 컬럼 목록]')
print(train_df.columns.tolist())

print('\n[test info]')
print(test_df.info())
print('\n[train info]')
print(train_df.info())

print('\n[test 결측치 개수 및 비율]')
null_summary = pd.DataFrame({
    'null_count': test_df.isnull().sum(),
    'null_ratio(%)': (test_df.isnull().mean() * 100).round(2),
})
print(null_summary[null_summary['null_count'] > 0])


print('\n[train 결측치 개수 및 비율]')
null_summary2 = pd.DataFrame({
    'null_count': train_df.isnull().sum(),
    'null_ratio(%)': (train_df.isnull().mean() * 100).round(2),
})
print(null_summary2[null_summary2['null_count'] > 0])

train shape: (600, 12)
test shape : (243, 11)

[test 컬럼 목록]
['title', 'distributor', 'genre', 'release_time', 'time', 'screening_rat', 'director', 'dir_prev_bfnum', 'dir_prev_num', 'num_staff', 'num_actor']

[train 컬럼 목록]
['title', 'distributor', 'genre', 'release_time', 'time', 'screening_rat', 'director', 'dir_prev_bfnum', 'dir_prev_num', 'num_staff', 'num_actor', 'box_off_num']

[test info]
<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           243 non-null    str    
 1   distributor     243 non-null    str    
 2   genre           243 non-null    str    
 3   release_time    243 non-null    str    
 4   time            243 non-null    int64  
 5   screening_rat   243 non-null    str    
 6   director        243 non-null    str    
 7   dir_prev_bfnum  107 non-null    float64
 8   dir_prev_num    243 non-null    int64  
 9   num_staff

# **전처리**

- dir_prev_bfnum 결측  -> 0 으로 채우고 has_prev 추가
- test time = 2015     -> 장르 중앙값(애니메이션 83분)으로 교체
- 198분·40분, dir_prev_num 6 -> 그대로 둔다 (범위 밖일 뿐 오류가 아니다)
- num_staff = 0        -> 그대로 둔다 (소규모 영화 신호)
- 감독                 -> 쓰지 않는다 (test 63% 미등장, 타깃 인코딩 시 RMSE 12% 악화)
- 배급사               -> 표기 정규화 후 빈도 인코딩
- 장르·관람등급         -> 원-핫
- 타깃                 -> log1p 로 학습하고 expm1 로 되돌린다


In [3]:
import re
import pandas as pd


def _norm_distributor(s):
    """`(주)NEW` 와 `NEW`, `시네마달`·`시네마 달`·`(주)시네마달` 을 같은 값으로 만든다.

    이렇게 정리하면 고유 배급사가 204 -> 191 로 줄고,
    같은 회사의 작품 수가 합쳐져 빈도 인코딩이 정확해진다.
    """
    s = re.sub(r'주|㈜|주식회사', '', str(s))
    return re.sub(r'[\s\-\.]', '', s).strip()


def clean(df, time_median=None):
    """원본을 건드리지 않고 정리본을 돌려준다.

    time_median: 장르별 상영시간 중앙값 (train 에서 구한 것). None 이면 보정하지 않는다.
    """
    df = df.copy()

    # 상영시간 입력 오류 — test 「슈퍼레이서 엔지」가 2015분(개봉연도)으로 적혀 있다.
    # 198분·40분은 train 범위(45~180) 밖이지만 실제로 가능한 값이라 건드리지 않는다.
    if time_median is not None:
        bad = df['time'] > 200
        if bad.any():
            df.loc[bad, 'time'] = df.loc[bad, 'genre'].map(time_median).values

    # 결측 100%가 dir_prev_num == 0 인 행이다. 값이 빠진 게 아니라 '이전작이 없다'는 뜻.
    df['has_prev'] = (df['dir_prev_num'] > 0).astype(int)
    df['dir_prev_bfnum'] = df['dir_prev_bfnum'].fillna(0)

    df['dist_norm'] = df['distributor'].map(_norm_distributor)
    return df


def make_xy(train, test):
    """학습용 X, y 와 test 용 X_test 를 함께 만든다.

    타깃을 쓰지 않는 변환만 하므로 train/test 를 함께 처리해도 누수가 아니다.
    (타깃 인코딩이나 스케일러를 쓰고 싶다면 반드시 폴드 안에서 fit 해야 한다.)
    """
    time_median = train.groupby('genre')['time'].median()
    tr = clean(train, time_median)
    te = clean(test, time_median)

    y = tr['box_off_num'].values
    full = pd.concat([tr.drop(columns=['box_off_num']), te], ignore_index=True)
    n = len(tr)

    X = pd.DataFrame(index=full.index)
    X['time'] = full['time']
    X['num_staff'] = full['num_staff']  # 0 은 소규모 영화 신호라 그대로 둔다
    X['num_actor'] = full['num_actor']
    X['dir_prev_num'] = full['dir_prev_num']
    X['dir_prev_bfnum'] = full['dir_prev_bfnum']
    X['has_prev'] = full['has_prev']

    release_dt = pd.to_datetime(full['release_time'])
    X['year'] = release_dt.dt.year
    X['month'] = release_dt.dt.month

    genre_dummies = pd.get_dummies(full['genre'], prefix='genre', dtype=int)
    rat_dummies = pd.get_dummies(full['screening_rat'], prefix='rat', dtype=int)
    X = pd.concat([X, genre_dummies, rat_dummies], axis=1)

    # 배급사는 '어느 회사인가'보다 '몇 편을 낸 회사인가'가 신호다.
    # 감독은 test 63% 가 train 에 없어서 쓰지 않는다.
    dist_counts = full['dist_norm'].value_counts()
    X['dist_freq'] = full['dist_norm'].map(dist_counts).values

    X_train = X.iloc[:n].reset_index(drop=True)
    X_test = X.iloc[n:].reset_index(drop=True)
    return X_train, y, X_test


def make_x(train, test):
    """X_test 만 필요할 때."""
    return make_xy(train, test)[2]

# **모델 정의**

In [4]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

N_SPLITS = 5
SEEDS = [42, 2024]
sub = pd.DataFrame({"title": test_df["title"]})

# 1. 전처리 데이터 생성
X_train, y_train, X_test = make_xy(train_df, test_df)

# 2. 모델 목록 (RF, ET는 log1p 학습 / Ridge, Lasso, ElasticNet은 원래 값 학습)
model_configs = [
    (
        "rf",
        RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=42
        ),
        True,
    ),
    (
        "et",
        ExtraTreesRegressor(
            n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=42
        ),
        True,
    ),
    (
        "ridge_raw",
        make_pipeline(StandardScaler(), Ridge(alpha=10.0, random_state=42)),
        False,
    ),
    (
        "lasso_raw",
        make_pipeline(
            StandardScaler(), Lasso(alpha=1000.0, max_iter=20000, random_state=42)
        ),
        False,
    ),
    (
        "elasticnet_raw",
        make_pipeline(
            StandardScaler(),
            ElasticNet(alpha=1000.0, l1_ratio=0.5, max_iter=20000, random_state=42),
        ),
        False,
    ),
]



# **공통 함수**

In [6]:
from sklearn.base import clone
from sklearn.model_selection import KFold


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def _set_seed(m, s):
    for key in ("random_state", "random_seed", "seed"):
        try:
            if key in m.get_params():
                m.set_params(**{key: s})
                return
        except Exception:
            pass


def evaluate(model, X, y, log=True, seeds=SEEDS, verbose=True):
    """5-fold 교차검증을 seeds 개수만큼 반복해 RMSE 평균을 낸다."""
    scores = []
    for s in seeds:
        for tr_i, va_i in KFold(N_SPLITS, shuffle=True, random_state=s).split(X):
            m = clone(model) if hasattr(model, "get_params") else model
            _set_seed(m, s)
            yt = np.log1p(y[tr_i]) if log else y[tr_i]
            m.fit(X.iloc[tr_i], yt)
            p = m.predict(X.iloc[va_i])
            if log:
                p = np.expm1(p)
            scores.append(rmse(y[va_i], np.clip(p, 0, None)))
    mu, sd = float(np.mean(scores)), float(np.std(scores))
    if verbose:
        name = model.__class__.__name__
        print(
            "{:<24s} RMSE {:>12,.0f}   (폴드 표준편차 {:>9,.0f}, {}-fold x {}seed)".format(
                name, mu, sd, N_SPLITS, len(seeds)
            )
        )
    return mu, sd


def fit_predict(model, X, y, X_test, log=True, seeds=SEEDS):
    """시드를 바꿔 가며 전체 train 으로 학습한 예측을 평균한다."""
    preds = []
    for s in seeds:
        m = clone(model) if hasattr(model, "get_params") else model
        _set_seed(m, s)
        m.fit(X, np.log1p(y) if log else y)
        p = m.predict(X_test)
        preds.append(np.expm1(p) if log else p)
    return np.clip(np.mean(preds, axis=0), 0, None)


def save_submission(pred, sub, path="submission.csv"):
    """제출 파일을 쓴다. 행 순서는 test 와 같다."""
    out = sub.copy()
    out["box_off_num"] = np.asarray(pred).round().astype(int)
    out.to_csv(path, index=False, encoding="utf-8-sig")
    print(
        "저장: {}  (중앙값 {:,.0f} / 평균 {:,.0f} / 최대 {:,.0f})".format(
            path,
            np.median(out.box_off_num),
            out.box_off_num.mean(),
            out.box_off_num.max(),
        )
    )
    return out

# **교차 검증**

In [7]:
# 3. 교차검증(CV) 평가 및 전체 학습 기반 테스트 예측
test_preds = {}
for name, model, use_log in model_configs:
    mu, sd = evaluate(model, X_train, y_train, log=use_log, seeds=SEEDS, verbose=False)
    print(f"[{name:14s}] 평균 RMSE: {mu:>12,.0f} (폴드 표준편차: {sd:>9,.0f})")
    test_preds[name] = fit_predict(
        model, X_train, y_train, X_test, log=use_log, seeds=SEEDS
    )

[rf            ] 평균 RMSE:    1,536,172 (폴드 표준편차:   301,549)
[et            ] 평균 RMSE:    1,560,342 (폴드 표준편차:   313,102)
[ridge_raw     ] 평균 RMSE:    1,503,870 (폴드 표준편차:   271,099)
[lasso_raw     ] 평균 RMSE:    1,505,486 (폴드 표준편차:   269,422)
[elasticnet_raw] 평균 RMSE:    1,791,977 (폴드 표준편차:   348,374)


# **제출 파일 생성**

In [8]:
# 4. 제출 파일 생성 (2개로 나눔)
save_submission(
    0.5 * test_preds["rf"] + 0.5 * test_preds["et"],
    sub,
    path="sub_na_01_rf_et_blend.csv",
)
save_submission(
    test_preds["ridge_raw"],
    sub,
    path="sub_na_02_ridge_raw.csv",
)

# 5. 테스트 예측값 저장 (Lasso, ElasticNet 포함)
pd.DataFrame(test_preds).to_csv("test_na_all_models.csv", index=False)

저장: sub_na_01_rf_et_blend.csv  (중앙값 11,433 / 평균 412,856 / 최대 6,532,876)
저장: sub_na_02_ridge_raw.csv  (중앙값 492,499 / 평균 845,608 / 최대 4,965,566)


In [9]:
combos = {
    "sub_na_03_lasso_raw.csv": test_preds["lasso_raw"],
    "sub_na_04_elasticnet_raw.csv": test_preds["elasticnet_raw"],
    "sub_na_05_ridge_lasso_blend.csv": 0.5 * test_preds["ridge_raw"]
    + 0.5 * test_preds["lasso_raw"],
    "sub_na_06_tree_linear_mix.csv": 0.5 * (0.5 * test_preds["rf"] + 0.5 * test_preds["et"])
    + 0.5 * (0.5 * test_preds["ridge_raw"] + 0.5 * test_preds["lasso_raw"]),
}

for fname, pred in combos.items():
  save_submission(pred, sub, path=fname)

저장: sub_na_03_lasso_raw.csv  (중앙값 475,855 / 평균 846,938 / 최대 5,002,030)
저장: sub_na_04_elasticnet_raw.csv  (중앙값 706,937 / 평균 708,426 / 최대 730,324)
저장: sub_na_05_ridge_lasso_blend.csv  (중앙값 484,177 / 평균 846,273 / 최대 4,983,798)
저장: sub_na_06_tree_linear_mix.csv  (중앙값 250,663 / 평균 629,564 / 최대 5,758,337)


In [10]:
# 트리 계열(0.5*rf + 0.5*et) 50% + Ridge(원래 값 학습) 50% 조합 제출 파일 생성
tree_pred = 0.5 * test_preds["rf"] + 0.5 * test_preds["et"]
tree_ridge_pred = 0.5 * tree_pred + 0.5 * test_preds["ridge_raw"]

save_submission(
    tree_ridge_pred,
    sub,
    path="sub_na_07_tree_ridge_mix.csv",
)

저장: sub_na_07_tree_ridge_mix.csv  (중앙값 254,824 / 평균 629,232 / 최대 5,749,221)


,title,box_off_num
0,용서는 없다,1401723
1,아빠가 여자를 좋아해,867612
2,하모니,1599239
3,의형제,1849283
4,평행 이론,1103334
...,...,...
238,해에게서 소년에게,219
239,울보 권투부,129958
240,어떤살인,856530
241,말하지 못한 비밀,7036


# **리더보드 결과**

### 리더보드 제출 결과 순위 (RMSE 기준)

| 순위 | 제출 ID | 파일명 | 리더보드 점수 (RMSE) | 제출 시각 | 비고 |
| :---: | :---: | :--- | :---: | :---: | :--- |
| **1위** | 1606497 | `sub_na_06_tree_linear_mix.csv` | **1,382,289.95** | 2026-11-27 22:44:36 | 트리(`0.5*RF + 0.5*ET`) + 선형(`0.5*Ridge + 0.5*Lasso`) |
| **2위** | 1606498 | `sub_na_07_tree_ridge_mix.csv` | **1,382,824.96** | 2026-11-27 22:50:08 | 트리(`0.5*RF + 0.5*ET`) + `Ridge` (50:50) |
| **3위** | 1606491 | `sub_na_01_rf_et_blend.csv` | **1,399,373.56** | 2026-11-27 22:32:41 | 트리 앙상블 단독 (`0.5*RF + 0.5*ET`, log1p 학습) |
| **4위** | 1606494 | `sub_na_03_lasso_raw.csv` | **1,461,314.74** | 2026-11-27 22:42:05 | `Lasso` 단독 (원래 값 학습) |
| **5위** | 1606496 | `sub_na_05_ridge_lasso_blend.csv` | **1,461,980.64** | 2026-11-27 22:43:41 | 선형 혼합 (`0.5*Ridge + 0.5*Lasso`) |
| **6위** | 1606492 | `sub_na_02_ridge_raw.csv` | **1,462,665.10** | 2026-11-27 22:33:28 | `Ridge` 단독 (원래 값 학습) |
| **7위** | 1606495 | `sub_na_04_elasticnet_raw.csv` | **1,855,909.75** | 2026-11-27 22:42:57 | `ElasticNet` 단독 (원래 값 학습) |